In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-01-small-data.py"

# 把 ESOL 的資料列數回可以比較的醇類

首次執行會下載 ESOL。若要修改分析，請編輯下一格的腳本，再依序執行該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors as D

# ESOL 水溶解度採 log10(mol/L)。原始文獻 DOI 10.1021/ci034243x。
_, (train, valid, test), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)

# 同一標準 SMILES 只計一次；此處只盤點結構，未合併任何量測。
structures = {Chem.MolToSmiles(Chem.MolFromSmiles(s))
              for part in (train, valid, test) for s in part.ids}
alcohols = []
for smiles in sorted(structures):
    mol = Chem.MolFromSmiles(smiles)
    atoms = list(mol.GetAtoms())
    oxygens = [a for a in atoms if a.GetAtomicNum() == 8]
    if (len(Chem.GetMolFrags(mol)) == 1 and mol.GetNumBonds() == len(atoms)-1
            and all(a.GetAtomicNum() in (6, 8) and a.GetFormalCharge() == 0 for a in atoms)
            and all(b.GetBondType() == Chem.BondType.SINGLE for b in mol.GetBonds())
            and len(oxygens) == 1 and oxygens[0].GetTotalNumHs() == 1):
        carbons = len(atoms) - 1
        alcohols.append((smiles, carbons))
counts = np.array([c for _, c in alcohols])
sizes, number = np.unique(counts, return_counts=True)
print("rows / unique structures / alcohols:", sum(map(len,(train, valid, test))),
      len(structures), len(alcohols))
print("C atoms / number of alcohols:", list(zip(sizes.tolist(), number.tolist())))
fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
ax.bar(sizes, number, color=["#ca662c" if k == 5 else "#387378" for k in sizes])
ax.set(xlabel="Carbon atoms per molecule", ylabel="Distinct ESOL structures",
       xticks=range(min(sizes), max(sizes)+1))
for k, n in zip(sizes, number):
    ax.text(k, n+0.14, str(n), ha="center", fontsize=9)
ax.set_ylim(0, max(number)+1.5)
fig.savefig(out/"12-01-small-data.png", dpi=170)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-01-small-data.png")))